# Used-car Risk & Anomaly Assessment, V2 experiments
Compare the preserved V1 baseline with a small, fixed experiment plan. All candidates use the same
training/validation/test partitions. Only validation results select models, features and calibration.
The baseline test set is diagnosed once; after selection is frozen, evaluate the final model once.
Candidate comparison tables are explicitly validation tables, so the test set never becomes a leaderboard.

Run top to bottom. The first run trains several CPU models and can take a while. Repeated runs reuse
local, data/split/parameter/version-verified caches in `.experiment_cache/`. Only the selected production
models and supporting statistics are saved under `artifacts/`. Delete the experiment cache to retrain.
No API or database connection is required. Scores remain transparent heuristics, not wrongdoing labels.


## 1. Imports and setup
Training runs on the CPU with a bounded thread count. Numeric missing values stay as NaN:
CatBoost handles them without inventing mileage, year or engine characteristics.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
import pickle
import gzip
import os
import platform

import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostRegressor

SEED = 42
FEATURES = ["brand", "model", "generation", "year", "mileage", "engine",
            "fuel_type", "gearbox", "drivetrain", "body_type"]
CATEGORICAL = ["brand", "model", "generation", "fuel_type", "gearbox", "drivetrain", "body_type"]
NUMERIC = ["year", "mileage", "engine"]
TARGET = "price_eur"
MISSING = "__MISSING__"
QUANTILES = np.array([0.10, 0.25, 0.50, 0.75, 0.90])
Q_NAMES = ["p10", "p25", "p50", "p75", "p90"]
MODEL_VERSION = "anomaly-risk-v2"
SCORING_POLICY_VERSION = "anomaly-risk-v2.3"
MODEL_PARAMS = dict(iterations=1000, depth=7, learning_rate=0.05,
                    loss_function="MultiQuantile:alpha=0.1,0.25,0.5,0.75,0.9",
                    random_seed=SEED, thread_count=min(4, os.cpu_count() or 1),
                    nan_mode="Min", allow_writing_files=False)

SCORING = {
    "weights": {"price": 0.75, "mileage": 0.10, "specification": 0.15},
    "risk_medium": 25.0, "risk_high": 50.0,
    "inner_score": 20.0, "outer_score": 60.0,
    "spread_floor": 1.0, "tail_width_fraction": 0.10,
    "mileage_min_samples": 30, "nearby_year_radius": 2,
    "spec_min_samples": 30, "engine_tolerance": 0.051,
    "spec_rare_frequency": 0.05, "spec_very_rare_frequency": 0.01,
    "spec_max_rarity_score": 60.0, "spec_outside_year_score": 80.0,
    "confidence_medium": 45.0, "confidence_high": 75.0,
    "confidence_model_support": 200, "confidence_generation_support": 100,
    "confidence_mileage_support": 100, "confidence_spec_support": 100,
    "confidence_relative_width_scale": 1.0,
    "support_thresholds": {"very_rare_max": 4, "rare_max": 14, "limited_max": 29},
    "rarity_penalties": {"rare_max": 8.0, "limited_max": 4.0},
}

# Support launching from either the notebook folder or the repository root.
BASE_DIR = next((p.resolve() for p in [Path.cwd(), Path.cwd() / "backend/ML_models/anomaly_risk"]
                 if (p / "anomaly_risk.ipynb").is_file() and (p / "data_ml.csv").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Run from anomaly_risk/ or the repository root, with data_ml.csv present.")
ARTIFACT_DIR = BASE_DIR / "artifacts"
print("Dataset folder:", BASE_DIR)
print("Package versions:", {p: version(p) for p in ["pandas", "numpy", "scipy", "catboost"]})
from time import perf_counter

REFERENCE_YEAR = 2026
MIN_SINGLE_IMPROVEMENT = 0.01
MIN_COMPLEX_IMPROVEMENT = 0.03
CACHE_DIR = BASE_DIR / ".experiment_cache"
CACHE_DIR.mkdir(exist_ok=True)
DATA_HASH = hashlib.sha256((BASE_DIR / "data_ml.csv").read_bytes()).hexdigest()
pd.set_option("display.max_columns", 24)


Dataset folder: D:\Code\python\999-project\backend\ML_models\anomaly_risk
Package versions: {'pandas': '3.0.5', 'numpy': '2.5.3', 'scipy': '1.18.1', 'catboost': '1.2.10'}


## 2. Load dataset
Only the ten specified vehicle features enter the price model. The target is always `price_eur`.

In [2]:
raw = pd.read_csv(BASE_DIR / "data_ml.csv")
required = FEATURES + [TARGET]
if set(raw.columns) != set(required):
    raise ValueError(f"Expected exactly {required}; got {raw.columns.tolist()}")
print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())

Shape: (59461, 11)
Columns: ['brand', 'model', 'generation', 'year', 'mileage', 'engine', 'fuel_type', 'gearbox', 'drivetrain', 'body_type', 'price_eur']


,brand,model,generation,year,mileage,engine,fuel_type,gearbox,drivetrain,body_type,price_eur
0,Mercedes,E-Class,W212 (2009 - 2016),2013,238000,2.2,Diesel,Automată,4x4,Sedan,14300.0
1,Mercedes,E-Class,W213 (2016 - 2023),2018,113000,2.0,Benzină,Automată,Din spate,Sedan,22500.0
2,Mercedes,E-Class,W213 (2016 - 2023),2019,187000,2.0,Diesel,Automată,Din spate,Sedan,24444.0
3,Mercedes,E-Class,W212 (2009 - 2016),2013,320458,2.2,Hybrid,Automată,Din spate,Sedan,16999.0
4,Mercedes,E-Class,W213 (2016 - 2023),2017,133000,2.0,Benzină,Automată,Din spate,Sedan,21500.0


## 3. Data inspection
Inspect actual categories and numeric ranges before interpreting results. No price cap is applied.

In [3]:
display(pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum()}))
display(raw.describe(include="number"))
display(raw[TARGET].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]))
display(raw[CATEGORICAL].nunique(dropna=True).rename("unique_values").to_frame())
for column in CATEGORICAL:
    print(column, "most frequent actual values:", raw[column].value_counts().head(5).to_dict())

,dtype,missing
brand,str,0
model,str,0
generation,str,62
year,int64,0
mileage,int64,0
engine,float64,183
fuel_type,str,0
gearbox,str,6
drivetrain,str,1
body_type,str,0


,year,mileage,engine,price_eur
count,59461.000000,59461.000000,59278.000000,59461.000000
mean,2014.538269,175216.063033,1.874539,16745.048235
std,7.352139,97380.117776,0.653002,16280.599802
min,1900.000000,70.000000,0.000000,200.000000
25%,2011.000000,108000.000000,1.500000,7000.000000
50%,2017.000000,170412.000000,1.900000,12900.000000
75%,2020.000000,235000.000000,2.000000,21000.000000
max,2026.000000,700000.000000,8.000000,299999.000000


count     59461.000000
mean      16745.048235
std       16280.599802
min         200.000000
1%          800.000000
5%         2200.000000
10%        3500.000000
25%        7000.000000
50%       12900.000000
75%       21000.000000
90%       32500.000000
95%       44900.000000
99%       79999.000000
max      299999.000000
Name: price_eur, dtype: float64

,unique_values
brand,121
model,1163
generation,1196
fuel_type,11
gearbox,5
drivetrain,4
body_type,16


brand most frequent actual values: {'BMW': 7166, 'Mercedes': 5543, 'Volkswagen': 4514, 'Toyota': 4321, 'Renault': 4311}
model most frequent actual values: {'5 Series': 1955, 'E-Class': 1494, 'X5': 1285, 'Focus': 1233, 'Megane': 1211}
generation most frequent actual values: {'I (2017 - prezent)': 1066, 'G30 (2017 - 2023)': 964, 'II (2012 - 2020)': 929, 'I (2015 - 2022)': 872, 'MK3 (2010 - 2018)': 789}
fuel_type most frequent actual values: {'Diesel': 26332, 'Benzină': 17625, 'Plug-in Hybrid (benzină)': 5885, 'Hybrid': 4547, 'Gaz / Benzină (propan)': 2180}
gearbox most frequent actual values: {'Automată': 39430, 'Mecanică': 17967, 'Variator': 1394, 'Robotizată': 659, 'Automat-Tiptronic': 5}
drivetrain most frequent actual values: {'Din față': 33479, '4x4': 19034, 'Din spate': 6935, '4x2': 12}
body_type most frequent actual values: {'Sedan': 14871, 'Crossover': 12462, 'SUV': 10696, 'Universal': 8244, 'Hatchback': 6459}


## 4. Data cleaning and preparation
Remove non-finite/non-positive targets and exact duplicate rows. Missing categories become
`__MISSING__`. Non-finite or negative numeric features become missing; non-positive or fractional
years also become missing. Zero engine size is retained, including for EVs. No mileage corrections
or price trimming are applied. The same feature preparation is used at inference.

In [4]:
def prepare_features(frame):
    result = frame.reindex(columns=FEATURES).copy()
    for column in CATEGORICAL:
        result[column] = result[column].astype("string").str.strip().replace("", pd.NA).fillna(MISSING).astype(str)
    for column in NUMERIC:
        values = pd.to_numeric(result[column], errors="coerce").astype(float)
        values = values.where(np.isfinite(values) & (values >= 0))
        if column == "year":
            values = values.where((values > 0) & (values % 1 == 0))
        result[column] = values
    return result

target = pd.to_numeric(raw[TARGET], errors="coerce")
valid_target = np.isfinite(target) & (target > 0)
clean = prepare_features(raw.loc[valid_target])
clean[TARGET] = target.loc[valid_target].astype(float)
duplicates = int(clean.duplicated().sum())
clean = clean.drop_duplicates().reset_index(drop=True)
print("Invalid/missing targets removed:", int((~valid_target).sum()))
print("Exact duplicate rows removed:", duplicates)
print("Rows after cleaning:", len(clean))
print("Missing numeric features after preparation:", clean[NUMERIC].isna().sum().to_dict())

Invalid/missing targets removed:

 0
Exact duplicate rows removed: 443
Rows after cleaning: 59018
Missing numeric features after preparation: {'year': 0, 'mileage': 0, 'engine': 183}


## 5. Reproducible train/test split
Use an approximately 80/20 random split with seed 42, reserving 10% of the training partition
for early stopping. Identical feature rows stay in one partition so repeated vehicle records
cannot straddle training and testing. This is a random snapshot evaluation, not a future-market
backtest; different records of the same vehicle cannot be identified without listing IDs.
All statistical reference tables use only the model-fitting partition.
The NumPy helper below reproduces the original GroupShuffleSplit algorithm exactly: sorted unique
groups, RandomState(42) permutation, ceil-sized holdout, then original row order. This avoids an
unneeded scikit-learn import chain; cached fits still require identical fit/validation row indices.

In [5]:
def group_split_indices(groups, test_size, random_state=42):
    unique, inverse = np.unique(np.asarray(groups), return_inverse=True)
    test_count = int(np.ceil(test_size * len(unique)))
    if not 0 < test_count < len(unique):
        raise ValueError("The split requires non-empty training and holdout groups")
    order = np.random.RandomState(random_state).permutation(len(unique))
    holdout = np.isin(inverse, order[:test_count])
    return np.flatnonzero(~holdout), np.flatnonzero(holdout)

feature_groups = pd.util.hash_pandas_object(clean[FEATURES], index=False)
train_idx, test_idx = group_split_indices(feature_groups, 0.20, SEED)
train_all = clean.iloc[train_idx].copy()
test = clean.iloc[test_idx].copy()
fit_idx, val_idx = group_split_indices(feature_groups.iloc[train_idx], 0.10, SEED)
train = train_all.iloc[fit_idx].copy()
validation = train_all.iloc[val_idx].copy()
assert set(feature_groups.loc[train.index]).isdisjoint(feature_groups.loc[test.index])
assert set(feature_groups.loc[validation.index]).isdisjoint(feature_groups.loc[test.index])
assert set(feature_groups.loc[train.index]).isdisjoint(feature_groups.loc[validation.index])
print(f"Train total: {len(train_all):,}; model fitting: {len(train):,}; validation: {len(validation):,}; test: {len(test):,}")

Train total: 47,239; model fitting: 42,528; validation: 4,711; test: 11,779


## 6. Baseline and error diagnosis
Preserve the original 999-tree V1 artifact when available; otherwise reproduce its fixed training
recipe. Diagnose actual-price bands, the 10 most common model/generation groups by **training** count,
and support buckets (including zero observations). Brand is always part of group keys.
These diagnostics do not change the predefined candidate list or selection thresholds.

In [6]:
def ordered_quantiles(predictions):
    values = np.asarray(predictions, dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Non-finite price prediction")
    return np.maximum(np.sort(values, axis=-1), SCORING["spread_floor"])

# Standard unweighted regression metrics, calculated directly with NumPy.
def regression_vectors(actual, predicted):
    y, p = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    if y.ndim != 1 or y.shape != p.shape or not y.size or not (np.isfinite(y).all() and np.isfinite(p).all()):
        raise ValueError("Expected matching finite non-empty 1D target/prediction arrays")
    return y, p

def mean_absolute_error(actual, predicted):
    y, p = regression_vectors(actual, predicted)
    return float(np.mean(np.abs(y - p)))

def mean_squared_error(actual, predicted):
    y, p = regression_vectors(actual, predicted)
    return float(np.mean((y - p) ** 2))

def r2_score(actual, predicted):
    y, p = regression_vectors(actual, predicted)
    residual = np.sum((y - p) ** 2)
    total = np.sum((y - np.mean(y)) ** 2)
    return float(1 - residual / total) if total > 0 else float(residual == 0)

def mean_pinball_loss(actual, predicted, alpha):
    y, p = regression_vectors(actual, predicted)
    residual = y - p
    return float(np.mean(np.maximum(alpha * residual, (alpha - 1) * residual)))

assert np.isclose(mean_absolute_error([1, 2, 3], [1, 2, 4]), 1 / 3)
assert np.isclose(mean_squared_error([1, 2, 3], [1, 2, 4]), 1 / 3)
assert np.isclose(r2_score([1, 2, 3], [1, 2, 4]), .5)
assert np.isclose(mean_pinball_loss([1, 2, 3], [1, 2, 4], alpha=.5), 1 / 6)

def price_metrics(actual, predicted):
    return {"P50_MAE": float(mean_absolute_error(actual, predicted)),
            "P50_RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
            "P50_R2": float(r2_score(actual, predicted))}

def quantile_metrics(actual, raw_q):
    q = ordered_quantiles(raw_q)
    y = np.asarray(actual)
    widths = q[:, 4] - q[:, 0]
    result = price_metrics(y, q[:, 2])
    result.update({f"{name.upper()}_coverage": float(np.mean(y <= q[:, i])) for i, name in enumerate(Q_NAMES)})
    result.update(P10_P90_coverage=float(np.mean((y >= q[:, 0]) & (y <= q[:, 4]))),
                  mean_interval_width=float(widths.mean()), median_interval_width=float(np.median(widths)),
                  relative_interval_width=float(np.mean(widths / q[:, 2])),
                  crossing_frequency=float(np.mean(np.any(np.diff(raw_q, axis=1) < 0, axis=1))),
                  mean_pinball_loss=float(np.mean([mean_pinball_loss(y, q[:, i], alpha=a)
                                                  for i, a in enumerate(QUANTILES)])))
    return result

baseline = CatBoostRegressor(**MODEL_PARAMS)
baseline_path = CACHE_DIR / "baseline_v1.cbm"
baseline_meta_path = CACHE_DIR / "baseline_v1_metadata.json"
baseline_meta = json.loads(baseline_meta_path.read_text(encoding="utf-8")) if baseline_meta_path.exists() else {}
baseline_valid = (baseline_meta.get("dataset_sha256") == DATA_HASH and
                  baseline_meta.get("packages", {}).get("catboost") == version("catboost") and
                  baseline_meta.get("train_fitting") == len(train) and baseline_meta.get("test") == len(test))
if baseline_path.exists() and baseline_valid:
    baseline.load_model(str(baseline_path))
else:
    baseline.fit(train[FEATURES], train[TARGET], cat_features=CATEGORICAL,
                 eval_set=(validation[FEATURES], validation[TARGET]), early_stopping_rounds=100,
                 use_best_model=True, verbose=200)
    baseline.save_model(str(baseline_path))
    baseline_meta_path.write_text(json.dumps({"dataset_sha256": DATA_HASH, "train_fitting": len(train),
        "test": len(test), "packages": {"catboost": version("catboost")}}), encoding="utf-8")

baseline_raw_test = baseline.predict(test[FEATURES])
baseline_test_q = ordered_quantiles(baseline_raw_test)
baseline_test_metrics = quantile_metrics(test[TARGET], baseline_raw_test)
baseline_val_raw = baseline.predict(validation[FEATURES])
baseline_val_metrics = quantile_metrics(validation[TARGET], baseline_val_raw)
support_counts = train.groupby(["brand", "model", "generation"]).size()

def error_diagnostics(frame, predictions):
    result = frame[FEATURES].copy()
    result["actual_price"] = frame[TARGET]
    result["predicted_p50"] = predictions[:, 2]
    result["absolute_error"] = abs(result["actual_price"] - result["predicted_p50"])
    result["training_support"] = [int(support_counts.get(tuple(row), 0))
                                  for row in result[["brand", "model", "generation"]].to_numpy()]
    result["price_band"] = pd.cut(result["actual_price"], [0, 5000, 10000, 20000, 30000, 50000, np.inf],
                                   labels=["under 5k", "5k-10k", "10k-20k", "20k-30k", "30k-50k", "50k+"], right=False)
    result["support_bucket"] = pd.cut(result["training_support"], [-1, 0, 4, 14, 29, np.inf],
                                       labels=["unseen (0)", "1-4", "5-14", "15-29", "30+"])
    return result

def error_summary(frame, keys):
    return frame.groupby(keys, observed=False).agg(rows=("absolute_error", "size"),
        MAE=("absolute_error", "mean"), total_absolute_error=("absolute_error", "sum"))

baseline_errors = error_diagnostics(test, baseline_test_q)
print("BASELINE test metrics:", baseline_test_metrics)
display(error_summary(baseline_errors, "price_band"))
common_keys = set(support_counts.nlargest(10).index)
common = baseline_errors[[tuple(row) in common_keys for row in baseline_errors[["brand", "model", "generation"]].to_numpy()]]
display(error_summary(common, ["brand", "model", "generation"]).sort_values("rows", ascending=False))
display(error_summary(baseline_errors, "support_bucket"))
display(baseline_errors.nlargest(20, "absolute_error")[FEATURES + ["actual_price", "predicted_p50", "absolute_error", "training_support"]])
diagnostic_tables = {"price_bands": error_summary(baseline_errors, "price_band").reset_index().to_dict(orient="records"),
                     "support_buckets": error_summary(baseline_errors, "support_bucket").reset_index().to_dict(orient="records")}
(CACHE_DIR / "baseline_diagnostics.json").write_text(json.dumps(diagnostic_tables, indent=2), encoding="utf-8")

BASELINE test metrics:

 {'P50_MAE': 2360.979057782846, 'P50_RMSE': 5833.089361009246, 'P50_R2': 0.8780108509476872, 'P10_coverage': 0.12225146447066813, 'P25_coverage': 0.2698870871890653, 'P50_coverage': 0.5033534255879107, 'P75_coverage': 0.7367348671364292, 'P90_coverage': 0.8816537906443671, 'P10_P90_coverage': 0.759402326173699, 'mean_interval_width': 6716.9076029534945, 'median_interval_width': 4200.67056300866, 'relative_interval_width': 0.46501256639030963, 'crossing_frequency': 0.01180066219543255, 'mean_pinball_loss': 851.7848340642755}


,rows,MAE,total_absolute_error
price_band,,,
under 5k,1858,863.063109,1.603571e+06
5k-10k,2719,1091.256280,2.967126e+06
10k-20k,3989,1663.659450,6.636338e+06
20k-30k,1808,2927.075174,5.292152e+06
30k-50k,963,5232.023119,5.038438e+06
50k+,442,14190.831485,6.272348e+06


,,,rows,MAE,total_absolute_error
brand,model,generation,,,
BMW,5 Series,G30 (2017 - 2023),199,2614.321493,520249.977030
Ford,Focus,MK3 (2010 - 2018),146,726.119960,106013.514199
Skoda,Superb,III (2015 - 2024),143,1955.991021,279706.716037
Renault,Kadjar,I (2015 - 2022),137,871.662662,119417.784713
BMW,X5,G05 (2018 - prezent),127,6026.608851,765379.324017
Volvo,XC90,II (2014 - prezent),111,3085.971746,342542.863757
Renault,Megane,IV (2016 - prezent),108,1000.224200,108024.213645
Peugeot,3008,II (2016 - 2023),102,1162.815011,118607.131131
Nissan,Qashqai,J11 (2013 - 2021),97,1915.872193,185839.602761


,rows,MAE,total_absolute_error
support_bucket,,,
unseen (0),96,6175.937730,5.928900e+05
1-4,478,3801.433550,1.817085e+06
5-14,945,2930.353826,2.769184e+06
15-29,1178,2725.314997,3.210421e+06
30+,9082,2138.338651,1.942039e+07


,brand,model,generation,year,mileage,engine,fuel_type,gearbox,drivetrain,body_type,actual_price,predicted_p50,absolute_error,training_support
52892,Ferrari,296,I (2021 - prezent),2024.0,2186.0,3.0,Plug-in Hybrid (benzină),Automată,4x4,Roadster,299999.0,80556.725702,219442.274298,0
52667,Bentley,Flying Spur,II (2019 - prezent),2024.0,11000.0,2.9,Plug-in Hybrid (benzină),Automată,4x4,Sedan,259900.0,78956.138074,180943.861926,1
55644,Land Rover,Range Rover,V (2021 - prezent),2023.0,9000.0,3.0,Diesel,Automată,4x4,SUV,260000.0,107000.922383,152999.077617,21
55643,Land Rover,Range Rover,V (2021 - prezent),2023.0,851.0,3.0,Diesel,Automată,4x4,SUV,229900.0,105718.392965,124181.607035,21
57571,Mercedes,G-Class,W463 (2018 - 2024),2020.0,98000.0,4.0,Benzină,Automată,4x4,SUV,210000.0,90575.946639,119424.053361,15
57859,Seat,Ibiza,IV (2008 - 2017),2011.0,243000.0,1.2,Diesel,Mecanică,Din față,Universal,123456.0,4241.469179,119214.530821,60
55627,Bentley,Bentayga,I (2015 - prezent),2022.0,49000.0,4.0,Benzină,Automată,4x4,SUV,219000.0,109728.024150,109271.975850,6
57857,Renault,Megane,III (2008 - 2016),2016.0,160000.0,1.5,Diesel,Automată,Din față,Universal,111500.0,8652.184465,102847.815535,247
57658,Land Rover,Range Rover,V (2021 - prezent),2023.0,65000.0,3.0,Diesel,Automată,4x4,SUV,179000.0,98196.515726,80803.484274,21
57845,Nissan,Qashqai,J11 (2013 - 2021),2014.0,197000.0,1.5,Diesel,Mecanică,Din față,Crossover,90900.0,10415.486771,80484.513229,428


1646

## 7. Controlled CatBoost experiments
Five fixed configurations span depths 6–10, learning rates 0.03–0.08 and L2 penalties 3–10.
Each has a maximum of 2,000 iterations and stops after 150 validation rounds without improvement.
CatBoost selects the best iteration using its objective; configurations are ranked primarily by validation
P50 MAE. No search framework or test-set feedback is used. Native CatBoost categorical handling remains.
See [CatBoost regression objectives](https://catboost.ai/docs/en/concepts/loss-functions-regression).

Local caches include dataset hash, exact fit/validation indices, feature transformations, parameters and
CatBoost version. They speed repeated notebook execution without putting experimental models in production.

In [7]:
EXPERIMENTS = [
    ("mq_d6_lr08_l2_3", 6, .08, 3),
    ("mq_d7_lr03_l2_5", 7, .03, 5),
    ("mq_d8_lr05_l2_5", 8, .05, 5),
    ("mq_d9_lr08_l2_10", 9, .08, 10),
    ("mq_d10_lr08_l2_10", 10, .08, 10),
]
experiment_rows = []

def model_frame(frame, derived=()):
    result = prepare_features(frame)
    age = REFERENCE_YEAR - result["year"]
    if "vehicle_age" in derived:
        result["vehicle_age"] = age
    if "mileage_per_year" in derived:
        result["mileage_per_year"] = result["mileage"] / age.clip(lower=1)
    if "engine_bucket" in derived:
        result["engine_bucket"] = (result["engine"] / .2).round() * .2
    return result

def fit_candidate(name, params, derived=()):
    spec = {"data_hash": DATA_HASH, "fit_indices": train.index.tolist(),
            "validation_indices": validation.index.tolist(), "params": params,
            "derived": list(derived), "reference_year": REFERENCE_YEAR,
            "catboost_version": version("catboost"), "preprocessing_version": "v2-1",
            "early_stopping_rounds": 150}
    key = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:20]
    path = CACHE_DIR / f"{name}_{key}.cbm"
    info_path = path.with_suffix(".json")
    candidate = CatBoostRegressor(**params)
    if path.exists() and info_path.exists():
        candidate.load_model(str(path))
        seconds = json.loads(info_path.read_text(encoding="utf-8"))["training_seconds"]
        print("Loaded cached candidate:", name)
    else:
        print("Training:", name, flush=True)
        start = perf_counter()
        candidate.fit(model_frame(train, derived), train[TARGET], cat_features=CATEGORICAL,
            eval_set=(model_frame(validation, derived), validation[TARGET]),
            early_stopping_rounds=150, use_best_model=True, verbose=400)
        seconds = perf_counter() - start
        candidate.save_model(str(path))
        info_path.write_text(json.dumps({"training_seconds": seconds, "parameters": params,
                                        "derived_features": list(derived), "cache_key": key}), encoding="utf-8")
    predicted = candidate.predict(model_frame(validation, derived))
    if predicted.ndim == 2:
        metric_values = quantile_metrics(validation[TARGET], predicted)
    elif params["loss_function"] in {"MAE", "Quantile:alpha=0.5"}:
        metric_values = price_metrics(validation[TARGET], np.maximum(predicted, SCORING["spread_floor"]))
    else:
        alpha = float(params["loss_function"].split("alpha=")[1])
        metric_values = {"alpha": alpha, "observed_coverage": float(np.mean(validation[TARGET] <= predicted)),
                         "pinball_loss": float(mean_pinball_loss(validation[TARGET], predicted, alpha=alpha))}
    if "P50_MAE" in metric_values:
        metric_values["MAE_improvement_vs_baseline_pct"] = 100 * (1 - metric_values["P50_MAE"] / baseline_val_metrics["P50_MAE"])
    experiment_rows.append({"configuration": name, **metric_values, "training_seconds": seconds,
                            "trees": candidate.tree_count_, "derived_features": list(derived)})
    (CACHE_DIR / "experiment_progress.json").write_text(json.dumps(experiment_rows, indent=2), encoding="utf-8")
    return {"name": name, "models": [candidate], "type": "multiquantile" if predicted.ndim == 2 else "median",
            "derived": list(derived), "params": [params], "validation_raw": predicted,
            "validation_metrics": metric_values}

baseline_bundle = {"name": "Baseline", "models": [baseline], "type": "multiquantile", "derived": [],
                   "params": [MODEL_PARAMS], "validation_raw": baseline_val_raw, "validation_metrics": baseline_val_metrics}
tuned_candidates = []
for name, depth, learning_rate, l2 in EXPERIMENTS:
    params = {**MODEL_PARAMS, "iterations": 2000, "depth": depth, "learning_rate": learning_rate, "l2_leaf_reg": l2}
    tuned_candidates.append(fit_candidate(name, params))
best_tuned = min(tuned_candidates, key=lambda candidate: candidate["validation_metrics"]["P50_MAE"])
display(pd.DataFrame([{"configuration": "Baseline", **baseline_val_metrics,
                       "MAE_improvement_vs_baseline_pct": 0.0}] + experiment_rows)[
    ["configuration", "P50_MAE", "P50_RMSE", "P50_R2", "MAE_improvement_vs_baseline_pct", "training_seconds", "trees"]])
print("Best tuned configuration by validation MAE:", best_tuned["name"])

Loaded cached candidate: mq_d6_lr08_l2_3
Loaded cached candidate: mq_d7_lr03_l2_5


Loaded cached candidate: mq_d8_lr05_l2_5
Loaded cached candidate: mq_d9_lr08_l2_10


Loaded cached candidate: mq_d10_lr08_l2_10


,configuration,P50_MAE,P50_RMSE,P50_R2,MAE_improvement_vs_baseline_pct,training_seconds,trees
0,Baseline,2255.568323,4667.214392,0.911835,0.000000,NaN,NaN
1,mq_d6_lr08_l2_3,2242.628542,4624.389540,0.913446,0.573682,228.776596,1727.0
2,mq_d7_lr03_l2_5,2238.505153,4672.462586,0.911637,0.756491,320.141323,1998.0
3,mq_d8_lr05_l2_5,2209.070867,4613.412823,0.913856,2.061452,338.045797,1472.0
4,mq_d9_lr08_l2_10,2220.884484,4590.069976,0.914726,1.537698,175.314736,589.0
5,mq_d10_lr08_l2_10,2239.502966,4566.186445,0.915611,0.712253,148.738661,319.0


Best tuned configuration by validation MAE: mq_d8_lr05_l2_5


## 8. Dedicated median and separate quantile models
Train a dedicated median model with MAE loss and five separate models with Quantile losses at
0.10, 0.25, 0.50, 0.75 and 0.90. Use the winning base-feature configuration and identical splits.
Compare both accuracy and interval behavior on validation. A dedicated median may replace the central
prediction in a MultiQuantile bundle only if the improvement justifies another model. Projection keeps
its P50 intact and moves other quantiles only when needed to enforce ordering.

In [8]:
median_params = {**best_tuned["params"][0], "loss_function": "MAE"}
dedicated = fit_candidate("dedicated_p50", median_params)
separate_parts = []
for alpha, name in zip(QUANTILES, Q_NAMES):
    params = {**best_tuned["params"][0], "loss_function": f"Quantile:alpha={alpha:g}"}
    separate_parts.append(fit_candidate(f"separate_{name}", params))
separate_raw = np.column_stack([part["validation_raw"] for part in separate_parts])
separate_bundle = {"name": "Separate quantiles", "models": [part["models"][0] for part in separate_parts],
                   "type": "separate", "derived": [], "params": [part["params"][0] for part in separate_parts],
                   "validation_raw": separate_raw, "validation_metrics": quantile_metrics(validation[TARGET], separate_raw)}

def project_around_median(q, central):
    result = ordered_quantiles(q)
    result[:, 2] = np.maximum(central, SCORING["spread_floor"])
    result[:, 1] = np.minimum(result[:, 1], result[:, 2])
    result[:, 0] = np.minimum(result[:, 0], result[:, 1])
    result[:, 3] = np.maximum(result[:, 3], result[:, 2])
    result[:, 4] = np.maximum(result[:, 4], result[:, 3])
    return result

hybrid_raw = project_around_median(best_tuned["validation_raw"], dedicated["validation_raw"])
hybrid_bundle = {"name": "MultiQuantile + dedicated P50", "models": best_tuned["models"] + dedicated["models"],
                 "type": "hybrid", "derived": [], "params": best_tuned["params"] + dedicated["params"],
                 "validation_raw": hybrid_raw, "validation_metrics": quantile_metrics(validation[TARGET], hybrid_raw)}
display(pd.DataFrame([{"model_version": b["name"], **b["validation_metrics"]}
                      for b in [baseline_bundle, best_tuned, dedicated, separate_bundle, hybrid_bundle]]))

Loaded cached candidate: dedicated_p50
Loaded cached candidate: separate_p10


Loaded cached candidate: separate_p25


Loaded cached candidate: separate_p50


Loaded cached candidate: separate_p75
Loaded cached candidate: separate_p90


,model_version,P50_MAE,P50_RMSE,P50_R2,P10_coverage,P25_coverage,P50_coverage,P75_coverage,P90_coverage,P10_P90_coverage,mean_interval_width,median_interval_width,relative_interval_width,crossing_frequency,mean_pinball_loss,MAE_improvement_vs_baseline_pct
0,Baseline,2255.568323,4667.214392,0.911835,0.121630,0.260030,0.500106,0.732965,0.883889,0.762259,6580.257524,4180.807682,0.459053,0.015708,806.558926,NaN
1,mq_d8_lr05_l2_5,2209.070867,4613.412823,0.913856,0.133305,0.274464,0.498408,0.720866,0.861601,0.728295,6074.631480,3936.484384,0.423257,0.033963,799.224048,2.061452
2,dedicated_p50,2226.634557,4646.717217,0.912608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.282771
3,Separate quantiles,2222.766723,4636.915871,0.912976,0.112927,0.271280,0.498196,0.719592,0.885375,0.772447,7054.567428,4363.215664,0.494049,0.094248,808.046193,NaN
4,MultiQuantile + dedicated P50,2226.634557,4646.717217,0.912608,0.132668,0.272766,0.499257,0.722352,0.861813,0.729145,6088.317284,3942.744062,0.426994,0.000000,798.891590,NaN


## 9. Derived-feature ablations and available extra fields
Reference year is fixed at **2026**. Test adding vehicle age, mileage/year (divide by max(age, 1)),
and a numeric 0.2-litre engine bucket, one at a time. Each is retained only if it improves validation
MAE by at least 1% over the current feature set. Raw features remain unchanged, and no transformation
uses price. Missing inputs remain missing.

Repo inspection: `backend/models.py` declares the following fields on `listings_cleaned`, but none
is in this CSV. The existing export has no ID, so joining a fresh export by vehicle features would
be ambiguous. A future single-query export should include the features together, rather than guessing joins.

| Existing field | Why it may help | Leakage / availability considerations |
|---|---|---|
| horsepower | Separates engine output and trims | Safe if scraped as a vehicle fact; audit missingness. |
| class | Shares segment information across rare models | Safe if taxonomy is independent of price. Current class builder copies `model_class.market_segment`. Audit ambiguous mappings. |
| state | Distinguishes condition | Use original listing value, not a price-based condition estimate. |
| seller_type | Dealer/private pricing patterns | Available at listing time; market/source bias may change. |
| offer_type | Sale vs exchange affects stated prices | Safe as scraped; current cleaner already filters to sale/exchange. |
| registration_country | Import/registration costs can affect price | Current cleaner filters to Moldova; may have little remaining variation. |
| doors, seats | Distinguishes body/configuration variants | Safe vehicle facts; audit sparse or incorrect values. |

No extra fields, target encodings, price-derived inputs or database calls are added in this experiment.

In [9]:
derived_current = best_tuned
derived_experiments = []
for feature in ["vehicle_age", "mileage_per_year", "engine_bucket"]:
    proposed = derived_current["derived"] + [feature]
    candidate = fit_candidate("derived_" + "_".join(proposed), best_tuned["params"][0], proposed)
    improvement = 1 - candidate["validation_metrics"]["P50_MAE"] / derived_current["validation_metrics"]["P50_MAE"]
    accepted = improvement >= MIN_SINGLE_IMPROVEMENT
    derived_experiments.append({"feature": feature, "tested_features": proposed,
                                "validation_MAE": candidate["validation_metrics"]["P50_MAE"],
                                "improvement_pct": 100 * improvement, "retained": accepted})
    if accepted:
        derived_current = candidate
display(pd.DataFrame(derived_experiments))
best_derived = derived_current

Loaded cached candidate: derived_vehicle_age
Loaded cached candidate: derived_mileage_per_year


Loaded cached candidate: derived_engine_bucket


,feature,tested_features,validation_MAE,improvement_pct,retained
0,vehicle_age,[vehicle_age],2208.190436,0.039855,False
1,mileage_per_year,[mileage_per_year],2222.446880,-0.605504,False
2,engine_bucket,[engine_bucket],2217.614896,-0.386770,False


## 10. Validation-only selection and outer-quantile calibration
Prefer a single model. A tuned or derived single model needs at least 1% validation MAE improvement
over baseline. A two- or five-model bundle needs at least 3% improvement over the chosen single model,
and must not worsen mean pinball loss by more than 2%. Thresholds are set before results are read.

Calibrate P10 and P90 from validation residuals normalized by each predicted interval's width:
`delta_q = quantile((actual - predicted_q) / max(P90 - P10, 1), q)`.
At inference add `delta_q * original_width` to each outer endpoint. Preserve P25/P50/P75 and enforce
ordering and positivity. The scaling avoids adding the same euro margin to cheap and expensive cars.
Validation is reused for selection and calibration, so this is empirical calibration, not a formal
independent conformal guarantee. No offsets or choices use test labels.

In [10]:
single_best = min([baseline_bundle, best_tuned, best_derived], key=lambda b: b["validation_metrics"]["P50_MAE"])
if 1 - single_best["validation_metrics"]["P50_MAE"] / baseline_val_metrics["P50_MAE"] < MIN_SINGLE_IMPROVEMENT:
    single_best = baseline_bundle
selected = single_best
selection_reasons = [f"Single-model candidate selected on validation MAE: {single_best['name']}."]
for candidate in sorted([hybrid_bundle, separate_bundle], key=lambda b: len(b["models"])):
    gain = 1 - candidate["validation_metrics"]["P50_MAE"] / selected["validation_metrics"]["P50_MAE"]
    pinball_ok = candidate["validation_metrics"]["mean_pinball_loss"] <= 1.02 * selected["validation_metrics"]["mean_pinball_loss"]
    if gain >= MIN_COMPLEX_IMPROVEMENT and pinball_ok:
        selected = candidate
        selection_reasons.append(f"Accepted {candidate['name']}: validation MAE improved by {gain:.1%} with acceptable pinball loss.")
    else:
        selection_reasons.append(f"Rejected added complexity of {candidate['name']}: MAE gain {gain:.1%}, pinball guard {pinball_ok}.")

def calibrate_outer_quantiles(actual, predictions):
    q = ordered_quantiles(predictions)
    width = np.maximum(q[:, 4] - q[:, 0], SCORING["spread_floor"])
    return {"p10_delta": float(np.quantile((np.asarray(actual) - q[:, 0]) / width, .10)),
            "p90_delta": float(np.quantile((np.asarray(actual) - q[:, 4]) / width, .90)),
            "method": "validation_scaled_outer_residual_quantiles", "validation_rows": len(actual)}

def apply_calibration(predictions, parameters):
    q = ordered_quantiles(predictions)
    width = np.maximum(q[:, 4] - q[:, 0], SCORING["spread_floor"])
    q[:, 0] = np.clip(q[:, 0] + parameters["p10_delta"] * width, SCORING["spread_floor"], q[:, 1])
    q[:, 4] = np.maximum(q[:, 4] + parameters["p90_delta"] * width, q[:, 3])
    return q

calibration_parameters = calibrate_outer_quantiles(validation[TARGET], selected["validation_raw"])
selected_val_q = apply_calibration(selected["validation_raw"], calibration_parameters)
selected_val_metrics = quantile_metrics(validation[TARGET], selected_val_q)
comparison_columns = ["model_version", "P50_MAE", "P50_RMSE", "P50_R2"] + [f"{q.upper()}_coverage" for q in Q_NAMES] + ["P10_P90_coverage", "mean_interval_width", "crossing_frequency"]
validation_comparison = pd.DataFrame([
    {"model_version": label, **bundle["validation_metrics"]} for label, bundle in [
        ("Baseline", baseline_bundle), ("Best tuned MultiQuantile", best_tuned),
        ("Dedicated P50 only", dedicated), ("Separate quantiles", separate_bundle),
        ("MultiQuantile + dedicated P50", hybrid_bundle), ("Best useful derived-feature version", best_derived)]
] + [{"model_version": "Final calibrated: " + selected["name"], **selected_val_metrics}])
print("VALIDATION comparison (not test results):")
display(validation_comparison[comparison_columns])
print("Selection frozen:", selected["name"], "Derived features:", selected["derived"])
print("Calibration parameters:", calibration_parameters)
print("\n".join(selection_reasons))
selection_record = {"chosen_name": selected["name"], "chosen_type": selected["type"],
                    "derived_features": selected["derived"], "parameters": selected["params"],
                    "calibration_parameters": calibration_parameters, "selection_reasons": selection_reasons,
                    "validation_metrics": selected_val_metrics, "dataset_sha256": DATA_HASH}
(CACHE_DIR / "selection_before_test.json").write_text(json.dumps(selection_record, indent=2), encoding="utf-8")

VALIDATION comparison (not test results):


,model_version,P50_MAE,P50_RMSE,P50_R2,P10_coverage,P25_coverage,P50_coverage,P75_coverage,P90_coverage,P10_P90_coverage,mean_interval_width,crossing_frequency
0,Baseline,2255.568323,4667.214392,0.911835,0.121630,0.260030,0.500106,0.732965,0.883889,0.762259,6580.257524,0.015708
1,Best tuned MultiQuantile,2209.070867,4613.412823,0.913856,0.133305,0.274464,0.498408,0.720866,0.861601,0.728295,6074.631480,0.033963
2,Dedicated P50 only,2226.634557,4646.717217,0.912608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Separate quantiles,2222.766723,4636.915871,0.912976,0.112927,0.271280,0.498196,0.719592,0.885375,0.772447,7054.567428,0.094248
4,MultiQuantile + dedicated P50,2226.634557,4646.717217,0.912608,0.132668,0.272766,0.499257,0.722352,0.861813,0.729145,6088.317284,0.000000
5,Best useful derived-feature version,2209.070867,4613.412823,0.913856,0.133305,0.274464,0.498408,0.720866,0.861601,0.728295,6074.631480,0.033963
6,Final calibrated: mq_d8_lr05_l2_5,2209.070867,4613.412823,0.913856,0.100191,0.274464,0.498408,0.720866,0.900021,0.800042,7171.537442,0.000000


Selection frozen: mq_d8_lr05_l2_5 Derived features: []
Calibration parameters: {'p10_delta': -0.0691616688752151, 'p90_delta': 0.1114099371229856, 'method': 'validation_scaled_outer_residual_quantiles', 'validation_rows': 4711}
Single-model candidate selected on validation MAE: mq_d8_lr05_l2_5.
Rejected added complexity of MultiQuantile + dedicated P50: MAE gain -0.8%, pinball guard True.
Rejected added complexity of Separate quantiles: MAE gain -0.6%, pinball guard True.


1590

## 11. One final held-out evaluation
All settings, features and calibration offsets are now frozen. Compare baseline and final predictions
on the original held-out rows. Do not change the selection after seeing this table. The candidate
leaderboard above uses validation, resolving the requirement to compare every approach without tuning
on the test set. Test metrics describe a random snapshot, not future-market performance.

In [11]:
def bundle_predict(bundle, frame):
    prepared = model_frame(frame, bundle["derived"])
    if bundle["type"] == "multiquantile":
        return bundle["models"][0].predict(prepared)
    if bundle["type"] == "separate":
        return np.column_stack([m.predict(prepared) for m in bundle["models"]])
    if bundle["type"] == "hybrid":
        return project_around_median(bundle["models"][0].predict(prepared), bundle["models"][1].predict(prepared))
    raise ValueError("Unsupported quantile bundle")

final_raw_test = bundle_predict(selected, test)
final_uncalibrated_q = ordered_quantiles(final_raw_test)
test_predictions = apply_calibration(final_raw_test, calibration_parameters)
final_test_metrics = quantile_metrics(test[TARGET], test_predictions)
final_test_metrics["raw_crossing_frequency"] = float(np.mean(np.any(np.diff(final_raw_test, axis=1) < 0, axis=1)))
test_comparison = pd.DataFrame([
    {"model_version": "Baseline", **baseline_test_metrics},
    {"model_version": "Final uncalibrated (fixed selection)", **quantile_metrics(test[TARGET], final_raw_test)},
    {"model_version": "Final calibrated (fixed selection)", **final_test_metrics},
])
display(test_comparison[comparison_columns + ["median_interval_width", "relative_interval_width"]])
calibration = pd.DataFrame({"quantile": [q.upper() for q in Q_NAMES], "expected_coverage": QUANTILES,
                           "observed_coverage": [final_test_metrics[f"{q.upper()}_coverage"] for q in Q_NAMES]})
calibration["difference"] = calibration["observed_coverage"] - calibration["expected_coverage"]
display(calibration)
final_errors = error_diagnostics(test, test_predictions)
support_comparison = error_summary(baseline_errors, "support_bucket")[["rows", "MAE"]].rename(columns={"MAE": "baseline_MAE"})
support_comparison["final_MAE"] = error_summary(final_errors, "support_bucket")["MAE"]
support_comparison["MAE_improvement_pct"] = 100 * (1 - support_comparison["final_MAE"] / support_comparison["baseline_MAE"])
display(support_comparison)
display(error_summary(final_errors, "price_band"))
unseen_examples = test[FEATURES].copy()
unseen_examples["actual_price"] = test[TARGET]
unseen_examples[[q.upper() for q in Q_NAMES]] = test_predictions
display(unseen_examples.sample(8, random_state=SEED))

def predict_quantiles(vehicle_features):
    return apply_calibration(bundle_predict(selected, pd.DataFrame([vehicle_features])), calibration_parameters)[0]

model = selected["models"][0]
metrics = {"mae": final_test_metrics["P50_MAE"], "rmse": final_test_metrics["P50_RMSE"], "r2": final_test_metrics["P50_R2"]}
widths = test_predictions[:, 4] - test_predictions[:, 0]

,model_version,P50_MAE,P50_RMSE,P50_R2,P10_coverage,P25_coverage,P50_coverage,P75_coverage,P90_coverage,P10_P90_coverage,mean_interval_width,crossing_frequency,median_interval_width,relative_interval_width
0,Baseline,2360.979058,5833.089361,0.878011,0.122251,0.269887,0.503353,0.736735,0.881654,0.759402,6716.907603,0.011801,4200.670563,0.465013
1,Final uncalibrated (fixed selection),2305.960189,5687.145380,0.884039,0.138552,0.279396,0.498090,0.723491,0.866457,0.727906,6191.532425,0.037779,3861.110851,0.437329
2,Final calibrated (fixed selection),2305.960189,5687.145380,0.884039,0.105102,0.279396,0.498090,0.723491,0.903302,0.798200,7309.127452,0.000000,4558.317838,0.516014


,quantile,expected_coverage,observed_coverage,difference
0,P10,0.10,0.105102,0.005102
1,P25,0.25,0.279396,0.029396
2,P50,0.50,0.498090,-0.001910
3,P75,0.75,0.723491,-0.026509
4,P90,0.90,0.903302,0.003302


,rows,baseline_MAE,final_MAE,MAE_improvement_pct
support_bucket,,,,
unseen (0),96,6175.937730,5888.142905,4.659937
1-4,478,3801.433550,3701.664764,2.624504
5-14,945,2930.353826,2853.487447,2.623109
15-29,1178,2725.314997,2625.051617,3.678965
30+,9082,2138.338651,2096.277379,1.967007


,rows,MAE,total_absolute_error
price_band,,,
under 5k,1858,858.235987,1.594602e+06
5k-10k,2719,1072.925394,2.917284e+06
10k-20k,3989,1641.023819,6.546044e+06
20k-30k,1808,2858.554708,5.168267e+06
30k-50k,963,5113.149141,4.923963e+06
50k+,442,13601.232835,6.011745e+06


,brand,model,generation,year,mileage,engine,fuel_type,gearbox,drivetrain,body_type,actual_price,P10,P25,P50,P75,P90
7835,Renault,Talisman,I (2016 - prezent),2017.0,157000.0,1.5,Diesel,Automată,Din față,Sedan,11500.0,9507.230241,10728.136075,11632.476659,12565.841817,13699.971206
51564,Volkswagen,Transporter,T5 (2003 - 2015),2005.0,296000.0,1.9,Diesel,Mecanică,Din față,Microvan,8000.0,3313.984001,4638.113392,5767.082465,6478.843712,8003.957794
57160,BMW,X5,G05 (2018 - prezent),2020.0,217935.0,3.0,Plug-in Hybrid (benzină),Automată,4x4,SUV,41900.0,30574.314411,35087.955708,38004.688781,41587.057025,45561.994077
17221,Volkswagen,Touareg,I (2002 - 2010),2005.0,285000.0,2.5,Diesel,Automată,4x4,Crossover,9000.0,6439.582000,7457.095511,8283.289484,8417.105803,9804.282855
48840,Opel,Astra,J (2009 - 2015),2011.0,208654.0,1.3,Diesel,Mecanică,Din față,Hatchback,4800.0,4277.143301,4765.760605,5136.758741,5672.644671,6179.378965
14558,Toyota,Rav 4,XA50 (2018 - 2025),2021.0,53704.0,2.5,Plug-in Hybrid (benzină),Automată,4x4,SUV,28499.0,26528.200163,28887.845001,30509.826365,33231.632977,36581.111981
22098,Audi,Q5,FY (2016 - prezent),2019.0,196393.0,2.0,Benzină,Automată,4x4,Crossover,18999.0,17863.677167,19698.879877,21173.094089,23288.196438,25945.122536
33468,Ford,Focus,MK3 (2010 - 2018),2015.0,178778.0,1.5,Diesel,Automată,Din față,Universal,7500.0,6981.131553,7567.493146,8083.046466,8884.032140,9704.867237


## 12. Price anomaly logic
Score 0 at P50, 20 at P25/P75 and 60 at P10/P90, with linear interpolation inside those
bands. Outside P10/P90, the score approaches 100 exponentially using the corresponding tail
spread (at least 10% of total interval width and €1). This makes the same euro deviation more
unusual in a narrow market. Degenerate quantile bands use the spread floor for numerical stability.
Direction changes only outside P10/P90. These are editable scoring choices, not learned labels.

In [12]:
def spread_anomaly_score(value, quantiles):
    lo, inner_lo, median, inner_hi, hi = map(float, quantiles)
    lower_side = value < median
    distance = abs(float(value) - median)
    floor = SCORING["spread_floor"]
    # Expand collapsed bands minimally so the score remains continuous, including at P50.
    inner_distance = max(median - inner_lo if lower_side else inner_hi - median, floor)
    outer_distance = max(median - lo if lower_side else hi - median, inner_distance + floor)
    if distance == 0:
        return 0.0
    if distance <= inner_distance:
        return float(SCORING["inner_score"] * distance / max(inner_distance, floor))
    if distance <= outer_distance:
        return float(SCORING["inner_score"] + (SCORING["outer_score"] - SCORING["inner_score"]) *
                     (distance - inner_distance) / max(outer_distance - inner_distance, floor))
    tail_spread = max(outer_distance - inner_distance, (hi - lo) * SCORING["tail_width_fraction"], floor)
    return float(SCORING["outer_score"] + (100 - SCORING["outer_score"]) *
                 (1 - np.exp(-(distance - outer_distance) / tail_spread)))

def analyze_price_anomaly(vehicle_features, actual_price):
    actual_price = float(actual_price)
    if not np.isfinite(actual_price) or actual_price <= 0:
        raise ValueError("price must be a finite positive asking price in EUR")
    q = predict_quantiles(vehicle_features)
    direction = "unusually_cheap" if actual_price < q[0] else "unusually_expensive" if actual_price > q[4] else "normal"
    reason = ("Asking price is below the predicted P10 price." if direction == "unusually_cheap" else
              "Asking price is above the predicted P90 price." if direction == "unusually_expensive" else
              "Asking price is close to the expected market median." if q[1] <= actual_price <= q[3] else
              "Asking price lies within P10-P90, outside the central P25-P75 band.")
    return {"actual_price": actual_price, **dict(zip(Q_NAMES, map(float, q))),
            "deviation_from_p50_pct": float(100 * (actual_price - q[2]) / q[2]),
            "direction": direction, "price_anomaly_score": spread_anomaly_score(actual_price, q), "reason": reason}

## 13. Mileage anomaly statistics
Use at least 30 valid mileage observations. Fallback: brand/model/generation/exact year,
then within ±2 years, then brand/model/generation, then brand/model. Brand is always part of the key
to avoid collisions between manufacturers. Unsupported mileage returns a null score and direction
`unknown`, not a fabricated normal result. Missing generation/year skips the corresponding levels.
Only aggregate counts and percentiles are saved, not complete training records.

In [13]:
MILEAGE_QUANTILES = [.05, .10, .25, .50, .75, .90, .95]
MILEAGE_NAMES = ["p05", "p10", "p25", "p50", "p75", "p90", "p95"]

def mileage_summary(values):
    values = values.dropna()
    return {"sample_size": int(len(values)),
            **dict(zip(MILEAGE_NAMES, map(float, values.quantile(MILEAGE_QUANTILES))))} if len(values) else None

def build_mileage_stats(reference):
    result = {name: {} for name in ["exact_year", "nearby_years", "model_generation", "model"]}
    known = reference[(reference["brand"] != MISSING) & (reference["model"] != MISSING)]
    for key, group in known.groupby(["brand", "model"]):
        summary = mileage_summary(group["mileage"])
        if summary:
            result["model"][key] = summary
    for key, group in known[known["generation"] != MISSING].groupby(["brand", "model", "generation"]):
        summary = mileage_summary(group["mileage"])
        if summary:
            result["model_generation"][key] = summary
        for year, year_group in group.dropna(subset=["year"]).groupby("year"):
            summary = mileage_summary(year_group["mileage"])
            if summary:
                result["exact_year"][key + (int(year),)] = summary
        radius = SCORING["nearby_year_radius"]
        candidate_years = {int(y) + offset for y in group["year"].dropna().unique()
                           for offset in range(-radius, radius + 1)}
        for year in candidate_years:
            summary = mileage_summary(group.loc[group["year"].between(year - radius, year + radius), "mileage"])
            if summary:
                result["nearby_years"][key + (year,)] = summary
    return result

mileage_stats = build_mileage_stats(train)

def normalized_vehicle(vehicle):
    return prepare_features(pd.DataFrame([vehicle])).iloc[0].to_dict()

def mileage_anomaly_score(actual, stats):
    """Score mileage more gently than price because driving patterns vary widely."""
    p05, p10, p25, p50, p75, p90, p95 = (float(stats[name]) for name in MILEAGE_NAMES)
    if actual >= p50:
        points = ((p50, 0.0), (p75, 10.0), (p90, 20.0), (p95, 40.0))
    else:
        points = ((p50, 0.0), (p25, 10.0), (p10, 20.0), (p05, 40.0))
    for (near, near_score), (far, far_score) in zip(points, points[1:]):
        if min(near, far) <= actual <= max(near, far):
            return float(near_score + (far_score - near_score) * abs(actual - near) / max(abs(far - near), SCORING["spread_floor"]))
    boundary, boundary_score = points[-1]
    tail_span = max(abs(p95 - p90) if actual >= p50 else abs(p10 - p05), SCORING["spread_floor"])
    return float(boundary_score + (100 - boundary_score) * (1 - np.exp(-abs(actual - boundary) / tail_span)))

def analyze_mileage_anomaly(vehicle):
    v = normalized_vehicle(vehicle)
    unknown = {"actual_mileage": None if pd.isna(v["mileage"]) else float(v["mileage"]),
               "expected_median_mileage": None, **{name: None for name in MILEAGE_NAMES},
               "mileage_anomaly_score": None, "direction": "unknown", "sample_size": 0,
               "comparison_level": "unsupported", "reason": "Insufficient mileage comparison data."}
    if pd.isna(v["mileage"]):
        return {**unknown, "reason": "Mileage is missing or invalid; no mileage score is available."}
    key = (v["brand"], v["model"], v["generation"])
    candidates = []
    if v["generation"] != MISSING:
        if pd.notna(v["year"]):
            candidates += [(name, key + (int(v["year"]),)) for name in ["exact_year", "nearby_years"]]
        candidates.append(("model_generation", key))
    candidates.append(("model", key[:2]))
    for level, group_key in candidates:
        stats = mileage_stats[level].get(group_key)
        if not stats or stats["sample_size"] < SCORING["mileage_min_samples"]:
            continue
        actual = float(v["mileage"])
        direction = "unusually_low" if actual < stats["p10"] else "unusually_high" if actual > stats["p90"] else "normal"
        reason = ("Mileage is unusually low relative to observed listings for similar vehicles." if direction == "unusually_low" else
                  "Mileage is unusually high relative to observed listings for similar vehicles." if direction == "unusually_high" else
                  "Mileage is within the observed P10-P90 range for similar vehicles.")
        if level in {"model_generation", "model"}:
            reason += " The fallback group mixes production years."
        return {"actual_mileage": actual, "expected_median_mileage": stats["p50"], **stats,
                "mileage_anomaly_score": mileage_anomaly_score(actual, stats),
                "direction": direction, "comparison_level": level, "reason": reason}
    return unknown

## 14. Specification rarity and inconsistency
Estimate each field separately within brand/model/generation. Engine values within 0.051 litres
count together, avoiding artificial rarity from tiny decimal differences. Known field observations
form each denominator; missing values are not treated as rare. Groups below 30 observations are
unsupported. An unseen value in a supported group is still only absent from this dataset.
Year outside the group's observed range is flagged separately from a rare year within it.
Use the largest supported field score so correlated attributes are not counted repeatedly.

In [14]:
SPEC_FIELDS = ["year", "engine", "fuel_type", "gearbox", "drivetrain", "body_type"]

def build_spec_stats(reference):
    result = {}
    known = reference[(reference[["brand", "model", "generation"]] != MISSING).all(axis=1)]
    for key, group in known.groupby(["brand", "model", "generation"]):
        fields = {}
        for field in SPEC_FIELDS:
            values = group[field].dropna()
            if field in CATEGORICAL:
                values = values[values != MISSING]
            fields[field] = {"sample_size": len(values), "counts": values.value_counts().to_dict()}
        result[key] = fields
    return result

spec_stats = build_spec_stats(train)

def analyze_specification(vehicle):
    v = normalized_vehicle(vehicle)
    group = spec_stats.get((v["brand"], v["model"], v["generation"]), {})
    signals, scores, supports = [], [], []
    for field in SPEC_FIELDS:
        value = v[field]
        data = group.get(field, {"sample_size": 0, "counts": {}})
        n, counts = int(data["sample_size"]), data["counts"]
        signal = {"field": field, "value": None if pd.isna(value) or value == MISSING else value,
                  "frequency": None, "sample_size": n, "severity": "unsupported", "score": None,
                  "reason": "Insufficient observations for this model and generation."}
        if signal["value"] is None:
            signal["reason"] = "The supplied field is missing or invalid."
        elif n >= SCORING["spec_min_samples"]:
            matches = (sum(count for observed, count in counts.items()
                           if abs(float(observed) - value) <= SCORING["engine_tolerance"])
                       if field == "engine" else counts.get(value, 0))
            frequency = float(matches / n)
            score = float(SCORING["spec_max_rarity_score"] * max(0, 1 - frequency / SCORING["spec_rare_frequency"]))
            severity, reason = "normal", "This value is observed regularly in the available listing data."
            if field == "year" and (value < min(counts) or value > max(counts)):
                severity, score = "outside_observed_range", SCORING["spec_outside_year_score"]
                reason = f"Year is outside the observed generation-year range {int(min(counts))}-{int(max(counts))}; verify the entry."
            elif matches == 0:
                severity = "unobserved"
                reason = "This value is unsupported by the observed values for this model and generation."
            elif frequency < SCORING["spec_rare_frequency"]:
                severity = "very_rare" if frequency < SCORING["spec_very_rare_frequency"] else "uncommon"
                reason = "This configuration is rarely observed in the available listing data."
            signal.update(frequency=frequency, severity=severity, score=score, reason=reason)
            scores.append(score)
            supports.append(n)
        signals.append(signal)
    return {"specification_anomaly_score": max(scores) if scores else None,
            "sample_size": min(supports) if supports else 0,
            "supported_fields": len(scores), "signals": signals}

## 15. Market confidence
Confidence is an evidence score, not risk or a calibrated probability. Combine bounded support
for model, generation, mileage and specifications with predicted interval precision. Cap confidence
for unseen categorical values, missing fields, sparse generations or entirely unobserved specs.
Observed group price dispersion is reported as context; the conditional model interval drives precision.

In [15]:
def build_market_stats(reference):
    def summarize(group):
        p25, p50, p75 = map(float, group[TARGET].quantile([.25, .50, .75]))
        return {"sample_size": len(group), "relative_price_iqr": (p75 - p25) / p50}
    return {
        "model": {key: summarize(g) for key, g in reference.groupby(["brand", "model"])},
        "generation": {key: summarize(g) for key, g in reference.groupby(["brand", "model", "generation"])},
        "known_categories": {col: set(reference[col]) - {MISSING} for col in CATEGORICAL},
    }

market_stats = build_market_stats(train)

def assess_market_confidence(vehicle, price_anomaly=None, mileage_anomaly=None, specification=None):
    v = normalized_vehicle(vehicle)
    if price_anomaly is None:
        price_anomaly = dict(zip(Q_NAMES, map(float, predict_quantiles(v))))
    if mileage_anomaly is None:
        mileage_anomaly = analyze_mileage_anomaly(v)
    if specification is None:
        specification = analyze_specification(v)
    key = (v["brand"], v["model"], v["generation"])
    model_support = market_stats["model"].get(key[:2], {}).get("sample_size", 0)
    gen_stats = market_stats["generation"].get(key, {}) if v["generation"] != MISSING else {}
    gen_support = gen_stats.get("sample_size", 0)
    width = float(price_anomaly["p90"] - price_anomaly["p10"])
    relative_width = width / max(price_anomaly["p50"], SCORING["spread_floor"])
    supports = [model_support, gen_support, mileage_anomaly["sample_size"], specification["sample_size"]]
    limits = [SCORING[f"confidence_{name}_support"] for name in ["model", "generation", "mileage", "spec"]]
    evidence = [min(n / limit, 1.0) for n, limit in zip(supports, limits)]
    evidence[3] *= specification["supported_fields"] / len(SPEC_FIELDS)
    precision = 1 / (1 + relative_width / SCORING["confidence_relative_width_scale"])
    score = 100 * (0.15 * evidence[0] + 0.25 * evidence[1] + 0.15 * evidence[2] + 0.15 * evidence[3] + 0.30 * precision)
    reasons = [f"{model_support} model observations; {gen_support} model and generation observations.",
               f"Predicted P10-P90 width is {width:.0f} EUR ({relative_width:.0%} of P50)."]
    unknown = [col for col in CATEGORICAL if v[col] != MISSING and v[col] not in market_stats["known_categories"][col]]
    missing = [col for col in FEATURES if pd.isna(v[col]) or v[col] == MISSING]
    if gen_support < SCORING["spec_min_samples"] or unknown:
        score = min(score, SCORING["confidence_medium"] - 1)
        reasons.append("Market confidence is low because this generation or categorical input has little training support.")
    if missing:
        score = min(score, SCORING["confidence_high"] - 1)
        reasons.append("Missing or invalid vehicle fields: " + ", ".join(missing) + ".")
    if any(s["severity"] in {"unobserved", "outside_observed_range"} for s in specification["signals"]):
        score = min(score, SCORING["confidence_high"] - 1)
        reasons.append("Some specifications fall outside observed support.")
    if mileage_anomaly["comparison_level"] in {"model_generation", "model", "unsupported"}:
        reasons.append("Mileage comparisons are broad or unavailable.")
    level = "high" if score >= SCORING["confidence_high"] else "medium" if score >= SCORING["confidence_medium"] else "low"
    return {"market_confidence": level, "confidence_score": float(score), "reasons": reasons,
            "model_observations": int(model_support), "generation_observations": int(gen_support),
            "p10_p90_width": width, "relative_interval_width": relative_width,
            "observed_relative_price_iqr": gen_stats.get("relative_price_iqr")}

## 16. Combined risk assessment
Weights: price 60%, mileage 25%, specification 15%. Scores below 25 are low, 25–49.99 medium,
and 50+ high. Missing components keep a null score and the available weights are renormalized;
the result reports which components were usable. Market confidence never enters this calculation.
`price` is the user's asking price in EUR, and is never included in the model features.

In [16]:
def assess_listing_risk(vehicle):
    if "price" not in vehicle:
        raise ValueError("Supply price as the asking price in EUR")
    price = analyze_price_anomaly(vehicle, vehicle["price"])
    mileage = analyze_mileage_anomaly(vehicle)
    specification = analyze_specification(vehicle)
    confidence = assess_market_confidence(vehicle, price, mileage, specification)
    scores = {"price": price["price_anomaly_score"], "mileage": mileage["mileage_anomaly_score"],
              "specification": specification["specification_anomaly_score"]}
    available = {name: score for name, score in scores.items() if score is not None}
    weight_sum = sum(SCORING["weights"][name] for name in available)
    effective_weights = {name: SCORING["weights"][name] / weight_sum for name in available}
    total = float(sum(effective_weights[name] * score for name, score in available.items()))
    level = "high" if total >= SCORING["risk_high"] else "medium" if total >= SCORING["risk_medium"] else "low"
    reasons = [price["reason"], mileage["reason"]]
    reasons += [f"{s['field']}: {s['reason']}" for s in specification["signals"] if s["severity"] != "normal"]
    reasons += confidence["reasons"]
    return {"scoring_policy_version": SCORING_POLICY_VERSION, "anomaly_score": total, "risk_level": level, "market_confidence": confidence["market_confidence"],
            "confidence_score": confidence["confidence_score"], "confidence": confidence,
            "components": {"price_anomaly": {**price, "score": price["price_anomaly_score"]},
                           "mileage_anomaly": {**mileage, "score": mileage["mileage_anomaly_score"]},
                           "specification_anomaly": {**specification, "score": specification["specification_anomaly_score"]}},
            "effective_weights": effective_weights, "reasons": reasons}

## 17. Unseen examples and pipeline checks
Print full results for real held-out listings, then change only the asking price of one held-out
vehicle. Assertions check price ordering, score behavior, missing data, unseen categories, statistical
fallbacks, exact feature isolation and JSON output. These are functional checks, not proof of market accuracy.

In [17]:
def test_vehicle(row):
    return {**normalized_vehicle(row.to_dict()), "price": float(row[TARGET])}

examples = [test_vehicle(row) for _, row in test.sample(3, random_state=SEED).iterrows()]
for example in examples:
    print("Vehicle:", json.dumps(example, ensure_ascii=False))
    print(json.dumps(assess_listing_risk(example), indent=2, ensure_ascii=False, allow_nan=False))

example = examples[0]
q = predict_quantiles(example)
assert q[0] > 1, "Choose another test vehicle with P10 above the prediction floor for the price demo"
price_scenarios = {"much_cheaper": float(q[0] * .3), "near_median": float(q[2]),
                   "much_higher": float(q[4] + 2 * max(q[4] - q[0], 1))}
scenario_results = {name: assess_listing_risk({**example, "price": price}) for name, price in price_scenarios.items()}
display(pd.DataFrame([{"scenario": name, "price": result["components"]["price_anomaly"]["actual_price"],
                       "price_score": result["components"]["price_anomaly"]["score"],
                       "direction": result["components"]["price_anomaly"]["direction"],
                       "anomaly_score": result["anomaly_score"], "confidence": result["market_confidence"]}
                      for name, result in scenario_results.items()]))

assert scenario_results["near_median"]["components"]["price_anomaly"]["score"] == 0
assert scenario_results["much_cheaper"]["components"]["price_anomaly"]["score"] > 60
assert scenario_results["much_higher"]["components"]["price_anomaly"]["score"] > 60
assert len({r["confidence_score"] for r in scenario_results.values()}) == 1
assert spread_anomaly_score(15000, [9000, 9500, 10000, 10500, 11000]) > spread_anomaly_score(15000, [1000, 5000, 10000, 15000, 19000])
assert spread_anomaly_score(10000.000001, [10000] * 5) < .001
assert spread_anomaly_score(10000, [10000] * 5) == 0
for boundary in q:
    assert abs(spread_anomaly_score(boundary - 1e-6, q) - spread_anomaly_score(boundary + 1e-6, q)) < .001
assert np.all(np.diff(test_predictions, axis=1) >= 0)
assert list(model.feature_names_) == list(model_frame(train.head(1), selected["derived"]).columns) and TARGET not in FEATURES and "price" not in FEATURES
assert np.array_equal(predict_quantiles(example), predict_quantiles({**example, "price": 999999, "Score": 100}))
assert analyze_mileage_anomaly({**example, "mileage": None})["mileage_anomaly_score"] is None
assert analyze_mileage_anomaly({**example, "mileage": -1})["mileage_anomaly_score"] is None
unknown = {**example, "brand": "__UNSEEN_TEST_BRAND__", "model": "__UNSEEN_TEST_MODEL__"}
unknown_result = assess_listing_risk(unknown)
assert unknown_result["market_confidence"] == "low"
assert unknown_result["components"]["specification_anomaly"]["score"] is None
assert unknown_result["components"]["mileage_anomaly"]["score"] is None
unknown_at_median = assess_listing_risk({**unknown, "price": float(predict_quantiles(unknown)[2])})
assert unknown_at_median["risk_level"] == "low" and unknown_at_median["market_confidence"] == "low"
json.dumps(unknown_result, allow_nan=False)
json.dumps(assess_listing_risk({"price": 10000}), allow_nan=False)
for invalid in [0, -1, float("nan"), float("inf")]:
    try:
        analyze_price_anomaly(example, invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid asking price was accepted")

# Select supported groups from real training values to exercise every fallback.
for level, table in mileage_stats.items():
    key, summary = next((k, s) for k, s in table.items() if s["sample_size"] >= SCORING["mileage_min_samples"])
    vehicle = {"brand": key[0], "model": key[1], "mileage": summary["p50"]}
    if level != "model":
        vehicle["generation"] = key[2]
    if level in {"exact_year", "nearby_years"}:
        vehicle["year"] = key[3]
    if level == "nearby_years":
        # Force a genuine sparse exact-year selection using another supported nearby group.
        key, summary = next((k, s) for k, s in table.items()
                            if s["sample_size"] >= SCORING["mileage_min_samples"] and
                            mileage_stats["exact_year"].get(k, {}).get("sample_size", 0) < SCORING["mileage_min_samples"])
        vehicle.update(brand=key[0], model=key[1], generation=key[2], year=key[3], mileage=summary["p50"])
    assert analyze_mileage_anomaly(vehicle)["comparison_level"] == level

spec_key, fields = next((k, s) for k, s in spec_stats.items() if s["year"]["sample_size"] >= SCORING["spec_min_samples"])
outside_year = analyze_specification(dict(zip(["brand", "model", "generation"], spec_key)) |
                                    {"year": max(fields["year"]["counts"]) + 10})
assert outside_year["signals"][0]["severity"] == "outside_observed_range"
engine_key, engine_fields = next((k, s) for k, s in spec_stats.items() if s["engine"]["sample_size"] >= SCORING["spec_min_samples"])
engine_vehicle = dict(zip(["brand", "model", "generation"], engine_key))
engine_value = max(engine_fields["engine"]["counts"], key=engine_fields["engine"]["counts"].get)
engine_exact = analyze_specification({**engine_vehicle, "engine": engine_value})["signals"][1]
engine_decimal = analyze_specification({**engine_vehicle, "engine": engine_value + 1e-6})["signals"][1]
assert engine_exact["frequency"] == engine_decimal["frequency"]
for result in list(scenario_results.values()) + [unknown_result]:
    assert 0 <= result["anomaly_score"] <= 100 and 0 <= result["confidence_score"] <= 100
    json.dumps(result, allow_nan=False)
print("Pipeline checks passed.")
assert np.array_equal(selected_val_q[:, 1:4], ordered_quantiles(selected["validation_raw"])[:, 1:4])
assert np.array_equal(test_predictions[:, 1:4], final_uncalibrated_q[:, 1:4])
assert not ({"price", "price_eur", "Score", "Deal Grade", "original_price"} & set(model.feature_names_))
assert np.allclose(predict_quantiles(example), apply_calibration(bundle_predict(selected, pd.DataFrame([example])), calibration_parameters)[0])
print("Calibration and selected-feature checks passed.")


Vehicle: {"brand": "Renault", "model": "Talisman", "generation": "I (2016 - prezent)", "year": 2017.0, "mileage": 157000.0, "engine": 1.5, "fuel_type": "Diesel", "gearbox": "Automată", "drivetrain": "Din față", "body_type": "Sedan", "price": 11500.0}
{
  "scoring_policy_version": "anomaly-risk-v2.3",
  "anomaly_score": 3.3799963105995356,
  "risk_level": "low",
  "market_confidence": "high",
  "confidence_score": 83.35178519907106,
  "confidence": {
    "market_confidence": "high",
    "confidence_score": 83.35178519907106,
    "reasons": [
      "148 model observations; 148 model and generation observations.",
      "Predicted P10-P90 width is 4193 EUR (36% of P50)."
    ],
    "model_observations": 148,
    "generation_observations": 148,
    "p10_p90_width": 4192.7409649647925,
    "relative_interval_width": 0.36043407502734764,
    "observed_relative_price_iqr": 0.21462389380530975
  },
  "components": {
    "price_anomaly": {
      "actual_price": 11500.0,
      "p10": 9507.230241

,scenario,price,price_score,direction,anomaly_score,confidence
0,much_cheaper,2852.169072,99.828306,unusually_cheap,76.053879,high
1,near_median,11632.476659,0.000000,normal,1.182650,high
2,much_higher,22085.453136,99.975397,unusually_expensive,76.164197,high


Pipeline checks passed.


Calibration and selected-feature checks passed.


## 18. Save the selected production artifacts and verify reload
Only the selected production model(s), original statistical artifacts and metadata belong in `artifacts/`.
Rejected candidates stay in the ignored experiment cache. Persist feature transformations, reference year,
calibration parameters, validation results and final test results so inference can reproduce the notebook.
Statistical risk and confidence functions are unchanged. Only load trusted local pickle files.

In [18]:
# V2.3 support-aware inference policy. It changes scoring policy only, not the trained CatBoost model.
def assess_market_support(vehicle):
        """Describe evidence for the exact brand, model and generation group."""
        v = normalized_vehicle(vehicle)
        key = (v["brand"], v["model"], v["generation"])
        generation = market_stats["generation"].get(key, {}) if v["generation"] != MISSING else {}
        observations = int(generation.get("sample_size", 0))
        thresholds = SCORING["support_thresholds"]
        penalties = SCORING["rarity_penalties"]
        if observations <= thresholds["very_rare_max"]:
            return {"model_generation_observations": observations, "support_level": "very_rare",
                    "rarity_penalty": 0.0}
        if observations <= thresholds["rare_max"]:
            # Join the limited-support curve without raising the penalty at 15.
            penalty = penalties["rare_max"] - (
                penalties["rare_max"] - penalties["limited_max"]
            ) * (observations - thresholds["very_rare_max"] - 1) / (
                thresholds["rare_max"] - thresholds["very_rare_max"] - 1
            )
            return {"model_generation_observations": observations, "support_level": "rare",
                    "rarity_penalty": round(penalty, 2)}
        if observations <= thresholds["limited_max"]:
            # 15 observations receives 4 points; 29 observations receives about 0.27.
            penalty = penalties["limited_max"] * (thresholds["limited_max"] + 1 - observations) / 15
            return {"model_generation_observations": observations, "support_level": "limited",
                    "rarity_penalty": round(penalty, 2)}
        return {"model_generation_observations": observations, "support_level": "normal",
                "rarity_penalty": 0.0}

def assess_market_confidence(vehicle, price_anomaly=None, mileage_anomaly=None,
                             specification=None, market_support=None):
        v = normalized_vehicle(vehicle)
        if price_anomaly is None:
            price_anomaly = dict(zip(Q_NAMES, map(float, predict_quantiles(v))))
        if mileage_anomaly is None:
            mileage_anomaly = analyze_mileage_anomaly(v)
        if specification is None:
            specification = analyze_specification(v)
        if market_support is None:
            market_support = assess_market_support(v)
        key = (v["brand"], v["model"], v["generation"])
        model_support = market_stats["model"].get(key[:2], {}).get("sample_size", 0)
        gen_stats = market_stats["generation"].get(key, {}) if v["generation"] != MISSING else {}
        gen_support = gen_stats.get("sample_size", 0)
        width = float(price_anomaly["p90"] - price_anomaly["p10"])
        relative_width = width / max(price_anomaly["p50"], SCORING["spread_floor"])
        supports = [model_support, gen_support, mileage_anomaly["sample_size"], specification["sample_size"]]
        limits = [SCORING[f"confidence_{name}_support"] for name in ["model", "generation", "mileage", "spec"]]
        evidence = [min(n / limit, 1.0) for n, limit in zip(supports, limits)]
        evidence[3] *= specification["supported_fields"] / len(SPEC_FIELDS)
        precision = 1 / (1 + relative_width / SCORING["confidence_relative_width_scale"])
        score = 100 * (0.15 * evidence[0] + 0.25 * evidence[1] + 0.15 * evidence[2] + 0.15 * evidence[3] + 0.30 * precision)
        reasons = [f"{model_support} model observations; {gen_support} model and generation observations.",
                   f"Predicted P10-P90 width is {width:.0f} EUR ({relative_width:.0%} of P50)."]
        unknown = [col for col in CATEGORICAL if v[col] != MISSING and v[col] not in market_stats["known_categories"][col]]
        missing = [col for col in FEATURES if pd.isna(v[col]) or v[col] == MISSING]
        if market_support["support_level"] == "very_rare":
            score = min(score, SCORING["confidence_medium"] - 1)
            reasons.append("Very limited market data is available for this model and generation.")
        elif market_support["support_level"] == "rare":
            score = min(score, SCORING["confidence_medium"] - 1)
            reasons.append("Market confidence is low because this model and generation have rare training support.")
        elif market_support["support_level"] == "limited":
            score = min(score, SCORING["confidence_high"] - 1)
            reasons.append("Market confidence is capped at medium because this model and generation have limited training support.")
        elif unknown:
            score = min(score, SCORING["confidence_medium"] - 1)
            reasons.append("Market confidence is low because one or more categorical inputs are outside training support.")
        if missing:
            score = min(score, SCORING["confidence_high"] - 1)
            reasons.append("Missing or invalid vehicle fields: " + ", ".join(missing) + ".")
        if any(s["severity"] in {"unobserved", "outside_observed_range"} for s in specification["signals"]):
            score = min(score, SCORING["confidence_high"] - 1)
            reasons.append("Some specifications fall outside observed support.")
        if mileage_anomaly["comparison_level"] in {"model_generation", "model", "unsupported"}:
            reasons.append("Mileage comparisons are broad or unavailable.")
        level = "high" if score >= SCORING["confidence_high"] else "medium" if score >= SCORING["confidence_medium"] else "low"
        return {"market_confidence": level, "confidence_score": float(score), "reasons": reasons,
                "model_observations": int(model_support), "generation_observations": int(gen_support),
                "p10_p90_width": width, "relative_interval_width": relative_width,
                "observed_relative_price_iqr": gen_stats.get("relative_price_iqr")}

def assess_listing_risk(vehicle):
        if "price" not in vehicle:
            raise ValueError("Supply price as the asking price in EUR")
        price = analyze_price_anomaly(vehicle, vehicle["price"])
        mileage = analyze_mileage_anomaly(vehicle)
        specification = analyze_specification(vehicle)
        market_support = assess_market_support(vehicle)
        confidence = assess_market_confidence(vehicle, price, mileage, specification, market_support)
        support_level = market_support["support_level"]
        assessment_status = "very_rare" if support_level == "very_rare" else (
            "limited_support" if support_level in {"rare", "limited"} else "full")
        message = None
        if support_level == "very_rare":
            message = ("Very limited market data is available for this model/generation, so a reliable overall "
                       "anomaly assessment cannot be produced.")
            reasons = [message, price["reason"], mileage["reason"]]
            reasons += [f"{s['field']}: {s['reason']}" for s in specification["signals"] if s["severity"] != "normal"]
            reasons += confidence["reasons"]
            return {"scoring_policy_version": SCORING_POLICY_VERSION, "assessment_status": assessment_status,
                    "market_support": market_support, "message": message, "anomaly_score": None, "risk_level": None,
                    "market_confidence": confidence["market_confidence"], "confidence_score": confidence["confidence_score"],
                    "confidence": confidence,
                    "components": {"price_anomaly": {**price, "score": price["price_anomaly_score"]},
                                   "mileage_anomaly": {**mileage, "score": mileage["mileage_anomaly_score"]},
                                   "specification_anomaly": {**specification, "score": specification["specification_anomaly_score"]}},
                    "effective_weights": {}, "reasons": reasons}
        scores = {"price": price["price_anomaly_score"], "mileage": mileage["mileage_anomaly_score"],
                  "specification": specification["specification_anomaly_score"]}
        available = {name: score for name, score in scores.items() if score is not None}
        weight_sum = sum(SCORING["weights"][name] for name in available)
        effective_weights = {name: SCORING["weights"][name] / weight_sum for name in available}
        total = min(100.0, float(sum(effective_weights[name] * score for name, score in available.items()) +
                                 market_support["rarity_penalty"]))
        level = "high" if total >= SCORING["risk_high"] else "medium" if total >= SCORING["risk_medium"] else "low"
        reasons = [price["reason"], mileage["reason"]]
        reasons += [f"{s['field']}: {s['reason']}" for s in specification["signals"] if s["severity"] != "normal"]
        reasons += confidence["reasons"]
        if market_support["rarity_penalty"]:
            reasons.append(f"A {market_support['rarity_penalty']:.2f}-point rarity adjustment was applied to the overall score.")
        return {"scoring_policy_version": SCORING_POLICY_VERSION, "assessment_status": assessment_status,
                "market_support": market_support, "message": message,
                "anomaly_score": total, "risk_level": level, "market_confidence": confidence["market_confidence"],
                "confidence_score": confidence["confidence_score"], "confidence": confidence,
                "components": {"price_anomaly": {**price, "score": price["price_anomaly_score"]},
                               "mileage_anomaly": {**mileage, "score": mileage["mileage_anomaly_score"]},
                               "specification_anomaly": {**specification, "score": specification["specification_anomaly_score"]}},
                "effective_weights": effective_weights, "reasons": reasons}

ARTIFACT_DIR.mkdir(exist_ok=True)
if selected["type"] == "separate":
    model_filenames = [f"price_{name}.cbm" for name in Q_NAMES]
elif selected["type"] == "hybrid":
    model_filenames = ["price_quantile_model.cbm", "price_p50.cbm"]
else:
    model_filenames = ["price_quantile_model.cbm"]
for fitted, filename in zip(selected["models"], model_filenames):
    fitted.save_model(str(ARTIFACT_DIR / filename))
for name, stats in [("mileage_stats", mileage_stats), ("spec_stats", spec_stats), ("market_stats", market_stats)]:
    with gzip.open(ARTIFACT_DIR / f"{name}.pkl.gz", "wb") as handle:
        pickle.dump(stats, handle, protocol=pickle.HIGHEST_PROTOCOL)
metadata = {
    "model_version": MODEL_VERSION, "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "chosen_model_type": selected["type"], "chosen_configuration": selected["name"], "model_files": model_filenames,
    "input_feature_names": FEATURES, "feature_names": list(model.feature_names_), "categorical_feature_names": CATEGORICAL,
    "numeric_feature_names": NUMERIC, "derived_features": selected["derived"], "reference_year": REFERENCE_YEAR,
    "derived_feature_formulas": {"vehicle_age": "reference_year - year", "mileage_per_year": "mileage / max(reference_year - year, 1)",
                                 "engine_bucket": "round(engine / 0.2) * 0.2"},
    "quantiles": QUANTILES.tolist(), "calibration_parameters": calibration_parameters,
    "model_parameters": selected["params"], "trees_retained": [m.tree_count_ for m in selected["models"]],
    "anomaly_scoring_constants": SCORING, "scoring_policy_version": SCORING_POLICY_VERSION, "missing_category": MISSING,
    "mileage_quantiles": MILEAGE_QUANTILES, "specification_fields": SPEC_FIELDS,
    "target": TARGET, "asking_price_field": "price", "currency": "EUR",
    "numeric_missing_strategy": "NaN, native CatBoost handling; negative/non-finite numeric values and non-positive/fractional years become missing",
    "prediction_postprocessing": "Sort positive quantiles; hybrid preserves dedicated median; apply outer calibration with monotonic bounds",
    "random_state": SEED, "rows_raw": len(raw), "rows_clean": len(clean), "train_total": len(train_all),
    "train_fitting": len(train), "validation": len(validation), "test": len(test),
    "statistics_reference": "model-fitting partition only", "dataset_sha256": DATA_HASH,
    "packages": {p: version(p) for p in ["pandas", "numpy", "scipy", "catboost"]},
    "validation_metrics": selected_val_metrics, "final_test_metrics": final_test_metrics,
    "baseline_test_metrics": baseline_test_metrics, "metrics": metrics,
    "calibration": calibration.to_dict(orient="records"), "selection_reasons": selection_reasons,
    "selection_thresholds": {"single_model_MAE_improvement": MIN_SINGLE_IMPROVEMENT, "added_models_MAE_improvement": MIN_COMPLEX_IMPROVEMENT},
}
metadata_path = ARTIFACT_DIR / "model_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2, allow_nan=False), encoding="utf-8")

before_reload = assess_listing_risk(example)
loaded = json.loads(metadata_path.read_text(encoding="utf-8"))
reloaded_models = []
for filename in loaded["model_files"]:
    restored = CatBoostRegressor()
    restored.load_model(str(ARTIFACT_DIR / filename))
    reloaded_models.append(restored)
selected = {**selected, "models": reloaded_models, "type": loaded["chosen_model_type"], "derived": loaded["derived_features"]}
REFERENCE_YEAR = loaded["reference_year"]
calibration_parameters = loaded["calibration_parameters"]
FEATURES, CATEGORICAL = loaded["input_feature_names"], loaded["categorical_feature_names"]
SCORING = loaded["anomaly_scoring_constants"]
with gzip.open(ARTIFACT_DIR / "mileage_stats.pkl.gz", "rb") as handle:
    mileage_stats = pickle.load(handle)
with gzip.open(ARTIFACT_DIR / "spec_stats.pkl.gz", "rb") as handle:
    spec_stats = pickle.load(handle)
with gzip.open(ARTIFACT_DIR / "market_stats.pkl.gz", "rb") as handle:
    market_stats = pickle.load(handle)
assert before_reload == assess_listing_risk(example)
print("Final artifact reload check passed.")

# Remove only known superseded production model names after successful reload verification.
known_model_names = {"price_quantile_model.cbm", *[f"price_{name}.cbm" for name in Q_NAMES]}
for filename in known_model_names - set(model_filenames):
    path = ARTIFACT_DIR / filename
    if path.exists():
        path.unlink()

for label, result in [("BASELINE", baseline_test_metrics), ("FINAL", final_test_metrics)]:
    print(f"{label}: MAE {result['P50_MAE']:,.2f} EUR; RMSE {result['P50_RMSE']:,.2f} EUR; "
          f"R² {result['P50_R2']:.4f}; P10-P90 coverage {result['P10_P90_coverage']:.2%}")
for name in ["P50_MAE", "P50_RMSE"]:
    change = baseline_test_metrics[name] - final_test_metrics[name]
    print(f"{name} reduction: {change:,.2f} EUR ({100 * change / baseline_test_metrics[name]:.2f}%)")
for name in ["P50_R2", "P10_P90_coverage"]:
    change = final_test_metrics[name] - baseline_test_metrics[name]
    print(f"{name} increase: {change:.4f} ({100 * change / baseline_test_metrics[name]:.2f}% relative)")
print(f"Final interval width: mean {widths.mean():,.2f} EUR, median {np.median(widths):,.2f} EUR")
print("Saved production artifacts:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(path, f"({path.stat().st_size:,} bytes)")

Final artifact reload check passed.
BASELINE: MAE 2,360.98 EUR; RMSE 5,833.09 EUR; R² 0.8780; P10-P90 coverage 75.94%
FINAL: MAE 2,305.96 EUR; RMSE 5,687.15 EUR; R² 0.8840; P10-P90 coverage 79.82%
P50_MAE reduction: 55.02 EUR (2.33%)
P50_RMSE reduction: 145.94 EUR (2.50%)
P50_R2 increase: 0.0060 (0.69% relative)
P10_P90_coverage increase: 0.0388 (5.11% relative)
Final interval width: mean 7,309.13 EUR, median 4,558.32 EUR
Saved production artifacts:
D:\Code\python\999-project\backend\ML_models\anomaly_risk\artifacts\market_stats.pkl.gz (49,404 bytes)
D:\Code\python\999-project\backend\ML_models\anomaly_risk\artifacts\mileage_stats.pkl.gz (666,785 bytes)
D:\Code\python\999-project\backend\ML_models\anomaly_risk\artifacts\model_metadata.json (7,261 bytes)
D:\Code\python\999-project\backend\ML_models\anomaly_risk\artifacts\price_quantile_model.cbm (22,191,544 bytes)
D:\Code\python\999-project\backend\ML_models\anomaly_risk\artifacts\spec_stats.pkl.gz (88,319 bytes)
